# Notebook to migrate regular 2D grid in .tiff format to CF compliant CoG's


### Import packages

In [187]:
import xarray as xr
import json

# Import custom functionality
from coclicodata.drive_config import p_drive
from coclicodata.etl.cf_compliancy_checker import check_compliancy, save_compliancy

### Define drive paths

In [188]:
processed_data_dir = p_drive.joinpath(r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\salinity")  

### Read raw data

In [189]:
# Define all scenarios
scenarios = ["baseline", "cc45y", "cc85y", "cc85sb2y", "cc45sm2y", "cc45sm2rb1y", "cc85sb2rb3y"] 

# Define the years (used as file names)
years = ["2018", "2030", "2040", "2050"]

probability = "p50"

# Create a dictionary to store all paths
ds_paths = {}

for scenario in scenarios:
    scenario_dir = processed_data_dir.joinpath(scenario)
    ds_paths[scenario] = {}

    for year in years:
        file_name = f"{probability}_{year}.tif"
        ds_paths[scenario][year] = scenario_dir.joinpath(file_name)

# Open all scenario/year datasets into a nested dictionary
ds_dic = {}

for scenario, year_paths in ds_paths.items():
    ds_dic[scenario] = {}
    for year, path in year_paths.items():
        try:
            ds_dic[scenario][year] = xr.open_dataset(path, engine="rasterio", mask_and_scale=False)
        except:
            print(f"The file {scenario}_{year} does not exist")


The file baseline_2030 does not exist
The file baseline_2040 does not exist
The file baseline_2050 does not exist
The file cc45y_2018 does not exist
The file cc85y_2018 does not exist
The file cc85sb2y_2018 does not exist
The file cc45sm2y_2018 does not exist
The file cc45sm2rb1y_2018 does not exist
The file cc85sb2rb3y_2018 does not exist
The file cc85sb2rb3y_2030 does not exist
The file cc85sb2rb3y_2040 does not exist
The file cc85sb2rb3y_2050 does not exist


### Check CF compliancy original NetCDF files

In [190]:
# Not implemented as geotiffs are less flexible, so checking compliance is not necessary.

### Make CF compliant alterations to the NetCDF files (dataset dependent)

In [191]:
# Not implemented

### Write data to CoG

#### Single CoG test

In [192]:
# Variables to include in a loop
VARIABLE = "salinity"
SCENARIO = "cc45y"
TIME = "2030"

ds = ds_dic[SCENARIO][TIME]

In [193]:
# Creating output folders
cog_dir  =  processed_data_dir.joinpath("cog")
cog_dir.mkdir(parents=True, exist_ok=True)

In [194]:
# Read metadata
metadata_path =  processed_data_dir.joinpath(f"metadata_{VARIABLE}.json")

# Attribute alterations by means of metadata template
f_global    = open(metadata_path)
meta_global = json.load(f_global)

# Add attributes to the dataset
for attr_name, attr_val in meta_global.items():
    if attr_name == 'PROVIDERS':
        attr_val = json.dumps(attr_val) #Line to include a dictionary as attribute
    if attr_name == "MEDIA_TYPE": # change media type to tiff, leave the rest as is
        attr_val = "IMAGE/TIFF"
    ds.attrs[attr_name] = attr_val

#TODO: Check if it applies to COG's
ds.attrs['Conventions'] = "CF-1.8"

In [195]:
# Process data
ds = ds.isel(band=0).drop_vars('band')
ds = ds.rio.write_crs(32648)
ds = ds.fillna(-99999)
ds.band_data.attrs['_FillValue'] = -99999


In [196]:
# Save COG 
output_dir  =  cog_dir.joinpath(SCENARIO)  # if 1x run, use cog dir, if multiple, use cogs dir
output_dir.mkdir(parents=True, exist_ok=True)

fname = f"{probability}_{TIME}.tif"
out_path = output_dir.joinpath(fname)
ds.rio.to_raster(out_path, compress="DEFLATE", driver="COG")

#### Multiple CoGs

In [197]:
# Variables to include in a loop
VARIABLE = ["salinity"] 
SCENARIO  = ["baseline", "cc45y", "cc85y", "cc85sb2y", "cc45sm2y", "cc45sm2rb1y", "cc85sb2rb3y"] 
TIME = ["2018","2030", "2040", "2050"]

In [198]:
## Creating output folders
cogs_dir =  processed_data_dir.joinpath("cogs")
cogs_dir.mkdir(parents=True, exist_ok=True)

In [199]:
## Loop over variables, scenarios and years:
for var in VARIABLE:

    # read metadata:
    metadata_path =  processed_data_dir.joinpath(f"metadata_{var}.json")
    # NetCDF attribute alterations by means of metadata template
    f_global    = open(metadata_path)
    meta_global = json.load(f_global)

    for scen in SCENARIO:
        print(f"Scenario:, {scen}")
        for time in TIME:
            print(f"  Year: {time}")
            try:
                # add all attributes (again)
                for attr_name, attr_val in meta_global.items():
                    if attr_name == 'PROVIDERS':
                        attr_val = json.dumps(attr_val)
                    if attr_name == "MEDIA_TYPE": # change media type to tiff, leave the rest as is
                        attr_val = "IMAGE/TIFF"
                    ds.attrs[attr_name] = attr_val

                ds.attrs['Conventions'] = "CF-1.8"

                # Process data
                ds = ds_dic[scen][time].isel(band=0).drop_vars('band')
                ds.rio.write_crs(32648, inplace=True)
                ds = ds.fillna(-99999)
                ds.band_data.attrs['_FillValue'] = -99999

                # Saving
                output_dir  =  cogs_dir.joinpath(scen)  # if 1x run, use cog dir, if multiple, use cogs dir
                output_dir.mkdir(parents=True, exist_ok=True)

                fname = f"{probability}_{time}.tif"
                out_path = output_dir.joinpath(fname)
                ds.rio.to_raster(out_path, compress="DEFLATE", driver="COG")
            except:
                print(f"     The file {scen}_{time} does not exist")

Scenario:, baseline
  Year: 2018
  Year: 2030
     The file baseline_2030 does not exist
  Year: 2040
     The file baseline_2040 does not exist
  Year: 2050
     The file baseline_2050 does not exist
Scenario:, cc45y
  Year: 2018
     The file cc45y_2018 does not exist
  Year: 2030
  Year: 2040
  Year: 2050
Scenario:, cc85y
  Year: 2018
     The file cc85y_2018 does not exist
  Year: 2030
  Year: 2040
  Year: 2050
Scenario:, cc85sb2y
  Year: 2018
     The file cc85sb2y_2018 does not exist
  Year: 2030
  Year: 2040
  Year: 2050
Scenario:, cc45sm2y
  Year: 2018
     The file cc45sm2y_2018 does not exist
  Year: 2030
  Year: 2040
  Year: 2050
Scenario:, cc45sm2rb1y
  Year: 2018
     The file cc45sm2rb1y_2018 does not exist
  Year: 2030
  Year: 2040
  Year: 2050
Scenario:, cc85sb2rb3y
  Year: 2018
     The file cc85sb2rb3y_2018 does not exist
  Year: 2030
     The file cc85sb2rb3y_2030 does not exist
  Year: 2040
     The file cc85sb2rb3y_2040 does not exist
  Year: 2050
     The file cc8